# 単純組成記述子によるベイズ探索

記述子は２変数以上を使わない（記述子が簡素なため）
目的変数はダミー値　１００回


In [ ]:
import itertools
import math
import numpy as np
import pandas as pd

seed = 123  # Fixed seed for reproducibility
n_open = 20  # Number of open observations to reveal
n_desc = 5


# ============================================================
# 1. Candidate generation
#    - 5 descriptors: conc_1 ... conc_5
#    - each takes values 0.0, 0.1, ..., 0.5
#    - at most 2 non-zero descriptors
# ============================================================

def generate_candidates(n_desc=5, max_digit=5, step=0.1, max_nonzero=2):
    rows = []

    for vals in itertools.product(range(max_digit + 1), repeat=n_desc):
        nonzero_count = sum(v != 0 for v in vals)

        if nonzero_count <= max_nonzero:
            row = [v * step for v in vals]
            rows.append(row)

    cols = [f"conc_{i+1}" for i in range(n_desc)]
    df = pd.DataFrame(rows, columns=cols)
    df.insert(0, "id", np.arange(len(df)))

    return df


# ============================================================
# 2. EASY dummy function parameter generator
#    - smoother
#    - weaker pair interaction
#    - fewer / broader bumps
# ============================================================

def make_effects_easy(df_candidates, conc_cols, seed=1, n_bumps=2):
    rng = np.random.RandomState(seed)
    n_desc = len(conc_cols)

    # Stronger monotonic component
    base_effect = rng.uniform(0.5, 1.5, size=n_desc)

    # Weaker pair interaction
    pair_strength = rng.uniform(-20.0, 20.0, size=(n_desc, n_desc))
    pair_strength = 0.5 * (pair_strength + pair_strength.T)
    np.fill_diagonal(pair_strength, 0.0)

    # Candidate compositions
    cand_x = df_candidates[conc_cols].values.astype(float)

    n_bumps = min(n_bumps, len(cand_x))
    bump_idx = rng.choice(len(cand_x), size=n_bumps, replace=False)
    bump_centers = cand_x[bump_idx].copy()

    # Fewer and broader peaks
    bump_heights = rng.uniform(10.0, 40.0, size=n_bumps)
    bump_widths = rng.uniform(0.15, 0.25, size=n_bumps)

    return {
        "base_effect": base_effect,
        "pair_strength": pair_strength,
        "bump_centers": bump_centers,
        "bump_heights": bump_heights,
        "bump_widths": bump_widths,
        "mode": "easy",
    }


# ============================================================
# 3. HARD dummy function parameter generator
#    - stronger interaction
#    - more local peaks
#    - one deceptive strong peak
# ============================================================

def make_effects_hard(df_candidates, conc_cols, seed=1, n_bumps=4):
    rng = np.random.RandomState(seed)
    n_desc = len(conc_cols)

    # Mixed-sign single-component effect
    base_effect = rng.uniform(-0.3, 1.2, size=n_desc)

    # Stronger pair interaction
    pair_strength = rng.uniform(-120.0, 120.0, size=(n_desc, n_desc))
    pair_strength = 0.5 * (pair_strength + pair_strength.T)
    np.fill_diagonal(pair_strength, 0.0)

    # Candidate compositions
    cand_x = df_candidates[conc_cols].values.astype(float)

    n_bumps = min(n_bumps, len(cand_x))
    bump_idx = rng.choice(len(cand_x), size=n_bumps, replace=False)
    bump_centers = cand_x[bump_idx].copy()

    bump_heights = rng.uniform(60.0, 180.0, size=n_bumps)
    bump_widths = rng.uniform(0.06, 0.16, size=n_bumps)

    # Add one deceptive strong peak in a not-obviously-good region
    lin_scores = cand_x @ np.maximum(base_effect, 0.0)
    hard_pool_size = max(10, len(cand_x) // 5)
    hard_pool = np.argsort(lin_scores)[:hard_pool_size]
    hard_idx = int(rng.choice(hard_pool))

    bump_centers[0] = cand_x[hard_idx]
    bump_heights[0] = rng.uniform(180.0, 260.0)
    bump_widths[0] = rng.uniform(0.05, 0.10)

    return {
        "base_effect": base_effect,
        "pair_strength": pair_strength,
        "bump_centers": bump_centers,
        "bump_heights": bump_heights,
        "bump_widths": bump_widths,
        "mode": "hard",
    }


# ============================================================
# 4. Raw dummy function
#    - behavior depends on effects["mode"]
# ============================================================

def dummy_true_function_raw(comp, effects):
    comp = np.asarray(comp, dtype=float)

    base_effect = effects["base_effect"]
    pair_strength = effects["pair_strength"]
    bump_centers = effects["bump_centers"]
    bump_heights = effects["bump_heights"]
    bump_widths = effects["bump_widths"]
    mode = effects["mode"]

    n_desc = len(comp)

    # ---- Base term ----
    lin = comp @ base_effect

    if mode == "easy":
        # Smoother and more monotonic
        base_term = 140.0 * np.tanh(1.8 * lin)
    else:
        # Harder: less dominant monotonic structure
        base_term = 120.0 * np.tanh(1.2 * lin)

    # ---- Pair term ----
    pair_term = 0.0
    for i in range(n_desc):
        for j in range(i + 1, n_desc):
            if comp[i] == 0.0 or comp[j] == 0.0:
                continue

            if mode == "easy":
                # Mild interaction
                pair_term += pair_strength[i, j] * comp[i] * comp[j]
                pair_term += 8.0 * comp[i] * comp[j] * math.cos(4.0 * (comp[i] - comp[j]))
            else:
                # Strong and oscillatory interaction
                pair_term += pair_strength[i, j] * comp[i] * comp[j]
                pair_term += 40.0 * comp[i] * comp[j] * math.cos(8.0 * (comp[i] - comp[j]))

    # ---- Local bumps ----
    bump_term = 0.0
    for center, h, w in zip(bump_centers, bump_heights, bump_widths):
        d2 = np.sum((comp - center) ** 2)
        bump_term += h * math.exp(-d2 / (2.0 * (w ** 2 + 1.0e-12)))

    return float(base_term + pair_term + bump_term)


# ============================================================
# 5. Scale raw values to [truth_min, truth_max]
# ============================================================

def scale_to_range(raw_vals, truth_min=100.0, truth_max=150.0):
    raw_vals = np.asarray(raw_vals, dtype=float)

    raw_min = raw_vals.min()
    raw_max = raw_vals.max()

    if abs(raw_max - raw_min) < 1.0e-12:
        return np.full_like(raw_vals, 0.5 * (truth_min + truth_max), dtype=float)

    scaled = truth_min + (raw_vals - raw_min) * (truth_max - truth_min) / (raw_max - raw_min)
    return scaled


# ============================================================
# 6. Add truth column from selected mode
# ============================================================

def add_truth_column(df_candidates, conc_cols, mode="easy", seed=1, truth_min=100.0, truth_max=150.0):
    df = df_candidates.copy()
    x = df[conc_cols].values.astype(float)

    if mode == "easy":
        effects = make_effects_easy(df, conc_cols, seed=seed, n_bumps=2)
    elif mode == "hard":
        effects = make_effects_hard(df, conc_cols, seed=seed, n_bumps=4)
    else:
        raise ValueError("mode must be 'easy' or 'hard'")

    raw_vals = np.array([dummy_true_function_raw(row, effects) for row in x])
    truth_vals = scale_to_range(raw_vals, truth_min=truth_min, truth_max=truth_max)

    return truth_vals


# ============================================================
# 7. Create open column
#    - randomly choose n_open rows
#    - copy truth into open
#    - others remain NaN
# ============================================================

def make_open_from_truth(truth_vals, n_open=20, seed=123):
    truth_vals = np.asarray(truth_vals, dtype=float)
    n_total = len(truth_vals)
    n_open = min(n_open, n_total)

    rng = np.random.RandomState(seed)
    open_idx = rng.choice(np.arange(n_total), size=n_open, replace=False)

    open_vals = np.full(n_total, np.nan, dtype=float)
    open_vals[open_idx] = truth_vals[open_idx]

    return open_vals, open_idx


# ============================================================
# 8. Main
# ============================================================

def main(seed=seed, n_open=20, n_desc=5):
    #n_desc = 5
    max_digit = 5
    step = 0.1
    max_nonzero = 2

    # Generate all candidates
    df_candidates = generate_candidates(
        n_desc=n_desc,
        max_digit=max_digit,
        step=step,
        max_nonzero=max_nonzero
    )

    conc_cols = [f"conc_{i+1}" for i in range(n_desc)]

    # Save candidate list only (Perl-like)
    df_candidates[conc_cols].to_csv("list.csv", index=False, header=False)

    # Base dataframe
    df_full = df_candidates.copy()

    # Easy version
    truth_easy = add_truth_column(
        df_candidates,
        conc_cols,
        mode="easy",
        seed=seed,
        truth_min=100.0,
        truth_max=150.0
    )

    open_easy, open_idx_easy = make_open_from_truth(
        truth_easy,
        n_open=n_open,
        seed=seed + 1000
    )

    df_full["truth_easy"] = truth_easy
    df_full["open_easy"] = open_easy

    # Hard version
    truth_hard = add_truth_column(
        df_candidates,
        conc_cols,
        mode="hard",
        seed=seed + 1,
        truth_min=100.0,
        truth_max=150.0
    )

    open_hard, open_idx_hard = make_open_from_truth(
        truth_hard,
        n_open=n_open,
        seed=seed + 2000
    )

    df_full["truth_hard"] = truth_hard
    df_full["open_hard"] = open_hard

    # Save all-in-one
    df_full.to_csv("candidates_easy_hard.csv", index=False)

    # Optional separate files
    df_easy = df_full[["id"] + conc_cols + ["truth_easy", "open_easy"]].copy()
    df_easy = df_easy.rename(columns={"truth_easy": "truth", "open_easy": "open"})
    df_easy.to_csv("candidates_easy.csv", index=False)

    df_hard = df_full[["id"] + conc_cols + ["truth_hard", "open_hard"]].copy()
    df_hard = df_hard.rename(columns={"truth_hard": "truth", "open_hard": "open"})
    df_hard.to_csv("candidates_hard.csv", index=False)

    print("Total candidates:", len(df_candidates))
    print("Saved:")
    print("  - list.csv")
    print("  - candidates_easy_hard.csv")
    print("  - candidates_easy.csv")
    print("  - candidates_hard.csv")
    print("")
    print("Easy:")
    print("  truth min =", np.min(truth_easy))
    print("  truth max =", np.max(truth_easy))
    print("  open count =", np.sum(~np.isnan(open_easy)))
    print("")
    print("Hard:")
    print("  truth min =", np.min(truth_hard))
    print("  truth max =", np.max(truth_hard))
    print("  open count =", np.sum(~np.isnan(open_hard)))


if __name__ == "__main__":
    main(seed=seed, n_open=n_open, n_desc=n_desc)